# Experiment v5 — Isolation Phase (Jupyter server)

Compute-budget rebuild of v4: **200-scenario** portfolio (100 single + 100 dual), **100 visits**,
ratios **{2, 5, 10, 20, 50}**, and a deterministic **STOP_COMPUTE** budget (thread-local
`ComputeMeter`) instead of wall-clock — so results no longer depend on CPU contention / threads.

**Prerequisites**
1. Repo cloned to the server, branch `v5-tests`.
2. JDK 8+ and Python 3 with pip.
3. Run from the `Tribes-master/` directory (set the path below).

**Notes**
- Linux classpath separator is `:` (Windows uses `;`).
- The full run is larger than v4 (200×100 turns × 12 configs × 3 budgets). Launch via `nohup` so it
  survives a disconnect; it is fully resumable per (agent, ratio, budget).
- Compute budgets default to `13400,26800,134000` (0.5x / 1x / 5x of the 1000ms-equiv 26800 units).

In [ ]:
import os
# EDIT to the absolute path of the Tribes-master directory on the server:
REPO = os.path.expanduser('~/Bachelor_Thesis/Tribes-master')
os.chdir(REPO)
print('cwd =', os.getcwd())
print(os.listdir('.'))   # expect src/, lib/, analysis/ ...

## 0. Environment check

In [ ]:
!java -version
!python --version
!nproc                 # number of cores
!free -g | head -2     # RAM in GB

## 1. Build test
Compiles the whole source tree to `out/`. Prints `BUILD OK` only if `javac` succeeds — if you see
`BUILD FAILED`, fix the compile error before running anything else.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 2. Smoke test (~seconds) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared random initial sequence across agents AND across
compute budgets (the heart of the paired methodology), folder layout, and CSV schema. If this
fails, do NOT launch the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV5

## 3. Run v5 (background + log)
Starts the full run detached so it survives a disconnect. Resumable per (agent, ratio, budget).

Args: `<threads> [<computeUnitsCsv>]`. Default budgets: `13400,26800,134000`.
32 GB heap is comfortable for ~120 in-flight tasks (each holds one GameState deep-copy).

In [ ]:
!nohup java -Xmx32g -cp "out:lib/*" core.data.exp.RunExperimentV5 120 > v5_run.log 2>&1 &
print('v5 started in background -> v5_run.log')

## 4. Monitor progress & thread use (re-run to refresh)
Shows how far the run has got (completed leaf CSVs out of 36 and the checkpoint count), the tail of
the log, and the live thread/CPU use of the run (`NLWP` = number of OS threads, `%CPU` = total CPU
across them — on a 120-thread run this should sit well above 100%).

In [ ]:
import subprocess
print('cores       :', subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
!echo "leaf CSVs   : $(find test_results/v5 -name results.csv 2>/dev/null | wc -l) / 36"
!echo "checkpoints : $(wc -l < test_results/v5/_completed.txt 2>/dev/null || echo 0) / 36 configs done"
print('--- run process: PID / threads (NLWP) / CPU% / elapsed ---')
!ps -o pid,nlwp,pcpu,etime,comm -C java 2>/dev/null || echo '(no java process running)'
!pgrep -af RunExperimentV5 || echo '(no RunExperimentV5 process - finished or not started)'
print('--- last 30 log lines ---')
!tail -n 30 v5_run.log 2>/dev/null || echo '(no log yet)'

## 5. Analyse (after the run completes)
`analyze_v5_full.py` produces the SAME tables and figures as v4 (T1–T8, fig3–fig8, E1–E16), pointed
at the v5 compute-budget data. Output: `analysis/figures_v5_full/` and `analysis/tables_v5_full/`.
Axis labels read compute units (`cu`) / compute budget rather than ms.

In [ ]:
!python -m pip install -q pandas matplotlib numpy scipy
!python analysis/analyze_v5_full.py test_results/v5

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('analysis/figures_v5_full/*.png')):
    print(f); display(Image(filename=f))

In [ ]:
import glob, pandas as pd
from IPython.display import display, Markdown
for f in sorted(glob.glob('analysis/tables_v5_full/*_summary.csv') + glob.glob('analysis/tables_v5_full/E*.csv')):
    display(Markdown(f'### {f}')); display(pd.read_csv(f, index_col=0))